In [ ]:
import gc

import cupy as cp
import torch

MIB = 1024 * 1024

In [ ]:
torch.cuda.init()
cp.cuda.runtime.memGetInfo()
torch.cuda.synchronize()
baseline_free = cp.cuda.runtime.memGetInfo()[0]


def held_mib():
    return (baseline_free - cp.cuda.runtime.memGetInfo()[0]) / MIB

In [ ]:
print("allocator:", "default")
print("baseline:", held_mib())

tensor = torch.empty(512 * MIB, dtype=torch.uint8, device="cuda")
print("torch allocated:", held_mib())

del tensor
gc.collect()
print("torch deleted:", held_mib())

array = cp.empty(512 * MIB, dtype=cp.uint8)
print("cupy allocated:", held_mib())

del array
gc.collect()
print("cupy deleted:", held_mib())

cp.get_default_memory_pool().free_all_blocks()
print("cupy free_all_blocks:", held_mib())